In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import ResNet50
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'tensorflow.python'

In [ ]:
dataset_name = "cats_vs_dogs"
(train_ds, val_ds, test_ds), dataset_info = tfds.load(dataset_name, split=["train[:70%]", "train[70%:85%]", "train[85%:]"], as_supervised=True, with_info=True)

In [ ]:
def format_data(image, label):
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (150, 150))
    image = tf.cast(image, tf.float32) / 255.0 
    return image, label

In [ ]:
train_ds = train_ds.map(format_data).batch(32).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(format_data).batch(32).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(format_data).batch(32).prefetch(tf.data.AUTOTUNE)

conv_base = ResNet50(weights='imagenet', include_top=False, input_shape=(150,150,3))
conv_base.trainable = False

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal", input_shape=(150, 150, 3)),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomWidth(0.2),
    layers.RandomHeight(0.2),
])

In [ ]:
inputs = keras.Input(shape=(150,150,3))
x = data_augmentation(inputs)
x = conv_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu',kernel_regularizer=regularizers.l2( l2=0.01))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="fine_tuning.keras",
        save_best_only=True,
        monitor="val_loss")
]

history = model.fit(train_ds, epochs=10, validation_data=val_ds)

In [ ]:
history = model.fit(train_ds, epochs=10, validation_data=val_ds)

conv_base.trainable = True
for layer in conv_base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-6), loss='binary_crossentropy', metrics=['accuracy'])

history_fine = model.fit(train_ds, epochs=10, validation_data=val_ds, callbacks=callbacks)

In [ ]:
conv_base.trainable = True
for layer in conv_base.layers[:-20]:
  layer.trainable = False

model.compile(optimizer=keras.optimizers.RMSprop(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="fine_tuning.keras",
        save_best_only=True,
        monitor="val_loss")
]
history_fin = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=callbacks)

In [ ]:
conv_base.trainable = True
for layer in conv_base.layers[:-10]:
  layer.trainable = False

model.compile(optimizer=keras.optimizers.SGD(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

history_fin = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=callbacks)

In [ ]:
model = keras.models.load_model("fine_tuning.keras")
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

In [ ]:
def plot_history(history, history_fine):
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Initial Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Initial Val Accuracy')
    plt.plot(history_fine.history['accuracy'], label='Fine-tune Train Accuracy')
    plt.plot(history_fine.history['val_accuracy'], label='Fine-tune Val Accuracy')
    plt.legend()
    plt.title('Training and Validation Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Initial Train Loss')
    plt.plot(history.history['val_loss'], label='Initial Val Loss')
    plt.plot(history_fine.history['loss'], label='Fine-tune Train Loss')
    plt.plot(history_fine.history['val_loss'], label='Fine-tune Val Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
    plt.show()

In [ ]:
plot_history(history, history_fin)